In [ ]:
import os

import rasterio

# Limite mémoire de la JVM r5py : DOIT être fixée via sys.argv AVANT le tout
# premier import de r5py (même `import r5py.util.jvm` déclenche l'exécution
# de r5py/__init__.py, qui importe des classes Java et démarre la JVM comme
# effet de bord — cf. r5py.util.jvm.start_jvm(), appelé par le hook d'import
# jpype dès qu'un package Java est touché). Une fois la JVM démarrée, son
# -Xmx est figé : réaffecter r5py.util.jvm.MAX_JVM_MEMORY après coup n'a
# AUCUN effet, silencieusement — vérifié : la JVM démarrait toujours avec le
# défaut de r5py (80% de la RAM totale, soit 12,8 Go sur une machine 16 Go),
# quelle que soit la valeur mise ici, ce qui explique les kernels morts en
# boucle lors de la construction du réseau régional. --max-memory (lu par
# r5py.util.config.Config via configargparse) est le seul levier qui marche.
# Remonter/descendre "8G" selon la RAM disponible (cf. kernel mort du 21/07
# et lors du run région Bretagne du 20/08).
import sys
sys.argv += ["--max-memory", "8G"]

import r5py
import r5py.util.jvm
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib
import branca as bc
import folium
import pyarrow
import pyarrow.parquet
#import r5py.sampledata.helsinki 

from src.info_reseau import dates_service, nom_reseau, nom_reseau_str
import gtfs_kit as gk

from src.utils import (
    charger_gtfs,
    longueur_lignes,
    km_par_ligne_jour,
    km_par_ligne_plage,
    obtenir_service_ids_pour_date,
    exporter_df_to_csv,
    exporter_geojson,
    exporter_gdf_to_csv,
    dir_tree,
    
)

from pathlib import Path
import datetime
import re
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx
import json
import time
import requests

from src.build_data_agglo import (
    codes_communes_via_api,
    details_communes,
    build_decoupage_agglo,
    decoupage_agglo_geojson,
    osm_pbf_creator,
    build_grid_agglo,
    build_grid_agglo_1km,
    fusionner_grille_resolution,
    ville_principale,
)

from src.BPE_traitement import (
    import_BPE, 
    filtre_BPE, 
    filtre_BPE_actifs, 
    carte_ponderation_domaine,
    mask_domaine_bpe,
   land_use_data_domaine,  
)

from src.ponderation_bpe import GAMMES_POIDS_PAR_DOMAINE, SEUILS_DOMAINE
from src.cartographie import (
    carte_population_infracommunale,
    echelle_continue_html,
    script_legende_en_bas,
    script_reajuster_si_masque,
    titre_carte_html,
)
from src.hf_cache import HF_DATA_REPO_ID_REGION, envoyer_vers_hf, fusionner_et_envoyer_csv, recuperer_depuis_hf

# calculer_ttm_par_lots : construction du ttm (par lots d'origines, écrit
# directement sur disque). calculer_index_benchmark_par_lots/
# cost_to_closest_par_lots/cumulative_cutoff_par_lots/deciles_niveau_vie :
# indicateurs et cartes d'accessibilité régionale, en lisant TTM_PATH par
# row group pyarrow plutôt qu'en chargeant le ttm entier en mémoire (cf. la
# docstring de calculer_index_benchmark_par_lots, src/utilitaires_matrix.py
# — la version "tout en mémoire" a déjà fait exploser la RAM à l'échelle
# région, même plafond que celui documenté sur IDFM à l'échelle agglo).
# gravity/decay_exponential/enhanced_2sfca(_par_lots) ne sont plus utilisés
# dans ce notebook (cellules exploratoires 3.2.x retirées, jamais adaptées
# par lots).
from src.utilitaires_matrix import (
    calculer_index_benchmark_par_lots,
    calculer_ttm_par_lots,
    cost_to_closest_par_lots,
    cumulative_cutoff_par_lots,
    deciles_niveau_vie,
)



In [2]:
#chemins fixe 

BASE_DIR = os.getcwd()  # Remonte d'un niveau depuis scripts/

BPE_XLS_PATH=os.path.join(BASE_DIR,'data','INSEE', "BPE_gammes_equipements_2025.xlsx")

MEMORY_CSV_AGGLO_DIR = os.path.join(BASE_DIR, "data", "memory_csv_agglo")
MEMORY_PBF_DIR = os.path.join(BASE_DIR, "data", "memory_pbf")
MEMORY_TTM_DIR = os.path.join(BASE_DIR, "data", "memory_ttm")

# Fond de carte pour tous les exports HTML interactifs (.explore()/DualMap) du
# notebook : change FOND_CARTE pour choisir parmi les trois options.
FONDS_CARTE = {
    "OpenStreetMap": "OpenStreetMap",
    "CartoDB Positron": "CartoDB positron",
    "CartoDB Dark Matter": "CartoDB dark_matter",
}
FOND_CARTE = "CartoDB Positron"  # "OpenStreetMap" / "CartoDB Positron" / "CartoDB Dark Matter"

output_path=os.path.join(BASE_DIR,'output')

data_path=os.path.join(BASE_DIR,'data')

# défini un zonage 1 km par 1 km 

In [3]:
#chemin GTFS 

GTFS_PATH=os.path.join(BASE_DIR,"data","GTFS_Region","Bretagne_KORRIGOBRET.gtfs.zip")


#charger le gtfs pour définir date 
feed = charger_gtfs(GTFS_PATH)

# Nom du réseau déterminé automatiquement à partir de la région desservie par
# le GTFS (cf. nom_region_via_gtfs, src/build_data_agglo.py) plutôt que codé
# en dur : nom_reseau_str(feed) (agency_name de agency.txt) ne convient pas
# ici, ce GTFS régional agrège 32 agences (STAR, BIBUS, TER...) et
# concaténerait leurs 32 noms plutôt que de donner un nom de réseau exploitable.
from src.build_data_agglo import nom_region_via_gtfs

NOM_REGION_STR = nom_region_via_gtfs(GTFS_PATH)
nom_reseau_str = NOM_REGION_STR

Chargement du fichier GTFS : /Users/antoinechevre/Documents/_0_Programme/3_Accessibilite_Region/data/GTFS_Region/Bretagne_KORRIGOBRET.gtfs.zip
✓ GTFS chargé avec succès
Région détectée via le GTFS : Bretagne (20/20 arrêt(s) échantillonné(s))


In [4]:
# créer un fichier decoupage_region.csv à partir du découpage administratif
# officiel de la région (geo.api.gouv.fr) : récupère TOUTES les communes de
# la région en 2 appels HTTP, indépendamment du GTFS utilisé pour l'analyse
# (contrairement à un géocodage arrêt par arrêt du GTFS, qui ne couvrirait
# que les communes desservies, et prendrait des heures sur un GTFS régional
# de dizaines de milliers d'arrêts, ex. TER Bretagne).
#
# NOM_REGION_STR déjà déterminé dans la cellule #chemin GTFS (à partir de
# GTFS_PATH) — pas recalculé ici.

from src.build_data_agglo import build_decoupage_region_via_api, decoupage_agglo_geojson

DECOUPAGE_REGION_PATH = os.path.join(data_path, "decoupage_region.csv")

decoupage_region = build_decoupage_region_via_api(
    nom_region=NOM_REGION_STR,
    output_path=DECOUPAGE_REGION_PATH,
)
decoupage_region

# créer un fichier geojson à partir de decoupage_region.csv

DECOUPAGE_REGION_PATH_GEOJSON = os.path.join(data_path, "decoupage_region.geojson")
decoupage_region_gdf = decoupage_agglo_geojson(
    csv_path=DECOUPAGE_REGION_PATH, output_path=DECOUPAGE_REGION_PATH_GEOJSON
)

✓ 1202 commune(s) écrite(s) dans /Users/antoinechevre/Documents/_0_Programme/3_Accessibilite_Region/data/decoupage_region.csv
✓ 1202 commune(s) écrite(s) dans /Users/antoinechevre/Documents/_0_Programme/3_Accessibilite_Region/data/decoupage_region.geojson


In [5]:
# créer un fichier grid 1x1km (carroyage Filosofi INSEE), directement à cette
# résolution — build_grid_agglo_1km récupère/vérifie la base carroyage 1km
# en local (assurer_carreaux_1km_local : repli sur le cache Hugging Face si
# absente) avant de la découper sur l'emprise de la région.

GRID_GPKG_PATH = os.path.join(data_path, f"population_grid_region_{NOM_REGION_STR}.gpkg")
population_grid_region = build_grid_agglo_1km(DECOUPAGE_REGION_PATH_GEOJSON, output_path=GRID_GPKG_PATH)

carreaux 1km dans l'agglo: 26251
population totale (ind): 3206210
ecrit dans: /Users/antoinechevre/Documents/_0_Programme/3_Accessibilite_Region/data/population_grid_region_Bretagne.gpkg


In [6]:
#import BPE

BPE_URL = "https://www.insee.fr/fr/statistiques/fichier/8217525/BPE25.parquet"

import_BPE(BPE_URL)

BPE25 déjà présent en local, pas de téléchargement : /Users/antoinechevre/Documents/_0_Programme/3_Accessibilite_Region/data/INSEE/BPE25.parquet


In [7]:
# BPE25.parquet (INSEE) est un parquet tabulaire classique (colonnes LONGITUDE/
# LATITUDE, pas de géométrie WKB/métadonnées GeoParquet) : gpd.read_parquet()
# échoue avec "Missing geo metadata". pd.read_parquet() est la bonne fonction ici.

land_use_data = population_grid_region[["id", "population"]].copy()

# decoupage_region.csv est plus simple que le .geojson ici : il suffit d'une jointure
# attributaire sur le code commune INSEE (pas besoin de jointure spatiale avec
# reprojection). code_insee est un int (ex: 17300) ; DEPCOM dans le BPE est une
# chaîne de 5 caractères (ex: "17300") : on caste et on zero-pad pour faire matcher.


# Liste des types d'équipements présents (TYPEQU = code le plus fin de la
# nomenclature BPE, ex: "C1", "D201"... ; le parquet ne contient pas les libellés
# associés, seulement les codes). DOM/SDOM donnent des catégories plus larges
# (domaine / sous-domaine) si TYPEQU est trop détaillé pour ton usage.


# Rattachement de BPE_region au carroyage population_grid_region : jointure spatiale.
# LAMBERT_X/LAMBERT_Y du BPE sont déjà en EPSG:2154 (vérifié : identique à la CRS
# de population_grid_region), donc pas besoin de reprojection.


BPE_region = filtre_BPE(DECOUPAGE_REGION_PATH, population_grid_region)


# Pondération de land_use_data par gamme d'équipement (BPE_gammes_equipements_2025.xlsx).
# Chaque TYPEQU appartient à une gamme (proximité/intermédiaire/supérieure/hors gamme),
# reflétant sa rareté/importance ; on pondère chaque équipement de BPE_region par le poids
# de sa gamme plutôt que de compter chaque équipement à l'identique.
#
# Note : ~2,4% des équipements de la zone (codes F1xx/F2xx/G10x, domaines
# sport/tourisme) n'ont pas de gamme dans la nomenclature officielle INSEE et
# sont donc exclus du score pondéré (mais restent comptés dans "TYPEQU" plus haut).

#proposition de pondération par gammes cf BPE_gammes_equipements_2025.xlsx dans le dossier data et file:///Users/antoinechevre/Desktop/Dossier_index/Data/BPE25_liste_hierarchisee_TYPEQU.html


recuperer_depuis_hf("BPE_gammes_equipements_2025.xlsx", BPE_XLS_PATH)

gamme_typequ = pd.read_excel(
    BPE_XLS_PATH,
    sheet_name="Gammes 2025 1 ligne 1 Typequ",
    header=4,
)[["TYPEQU", "GAMME"]]

BPE_region = BPE_region.merge(gamme_typequ, on="TYPEQU", how="left")

# Poids par gamme spécifique à chaque grand domaine BPE, importés de
# src/ponderation_bpe.py (cf. ce module pour le détail par domaine).
gammes_poids_par_domaine = GAMMES_POIDS_PAR_DOMAINE

table_poids_domaine_gamme = pd.DataFrame(
    [
        {"domaine": domaine, "GAMME": gamme, "poids_gamme": poids}
        for domaine, poids_par_gamme in gammes_poids_par_domaine.items()
        for gamme, poids in poids_par_gamme.items()
    ]
)

BPE_region["domaine"] = BPE_region["TYPEQU"].str[0]
BPE_region = BPE_region.merge(table_poids_domaine_gamme, on=["domaine", "GAMME"], how="left")

equipements_pondere_par_carreau = (
    BPE_region.dropna(subset=["id_carreau", "poids_gamme"])
    .groupby("id_carreau")["poids_gamme"]
    .sum()
)

land_use_data["equipements_pondere"] = (
    land_use_data["id"].map(equipements_pondere_par_carreau).fillna(0.0)
)

# Ne garder que les carreaux "actifs" pour l'analyse BPE : ceux qui ont de la
# population ou au moins un équipement pondéré (equipements_pondere, calculé
# ci-dessus). Les carreaux vides (ni habitants ni équipement) n'apportent rien
# aux cartes/calculs suivants.

population_grid_region = filtre_BPE_actifs(population_grid_region, land_use_data)

# On restreint aussi land_use_data aux mêmes carreaux actifs (mêmes id que
# population_grid_region ci-dessus) : sans ça, les moyennes/seuils "pôles"
# ci-dessous seraient tirés vers le bas par des milliers de carreaux vides
# (ni population ni équipement), ce qui fausserait le seuil pour les carreaux
# qui comptent réellement.
land_use_data = land_use_data[land_use_data["id"].isin(population_grid_region["id"])].reset_index(drop=True)

# Filtres land_use_data par domaine BPE (source :
# BPE25_liste_hierarchisee_TYPEQU.html) : un DataFrame land_use_data_<lettre>
# par domaine, avec le nombre d'équipements de ce domaine par carreau.
# TYPEQU commence toujours par la lettre du domaine (ex: "C107" -> domaine C),
# donc un simple str.startswith(lettre) suffit.
DOMAINES_BPE = {
    "O": "Tout équipements pondérés",
    "A": "Services pour les particuliers",
    "B": "Commerces",
    "C": "Enseignement",
    "D": "Santé et action sociale",
    "E": "Transports et déplacements",
    "F": "Sports, loisirs et culture",
    "G": "Tourisme",
}

# SEUILS_DOMAINE importé de src/ponderation_bpe.py (cf. cellule d'imports) :
# par domaine BPE, carreaux dont le score pondéré du
# domaine dépasse SEUILS_DOMAINE[domaine] fois la moyenne des carreaux de ce
# domaine (au lieu d'un seul seuil global sur le total pondéré "O").

seuils_equipements_pondere_par_domaine = {}

for d, seuil_pct in SEUILS_DOMAINE.items():
    valeurs_domaine = land_use_data_domaine(BPE_region, land_use_data, d)
    seuil = seuil_pct * valeurs_domaine[d].mean()
    seuils_equipements_pondere_par_domaine[d] = seuil

    land_use_data[f"pole_equipements_{d}"] = (valeurs_domaine[d] > seuil).astype(int)

    nb_carreaux_pole = land_use_data[f"pole_equipements_{d}"].sum()
    pct_carreaux_pole = 100 * nb_carreaux_pole / len(land_use_data)

    print(
        f"{nb_carreaux_pole} carreaux au-dessus du seuil ({pct_carreaux_pole:.1f}% des carreaux) "
        f"pour {DOMAINES_BPE.get(d, d)} ({seuil:.1f}, soit {seuil_pct:.0%} de la moyenne)"
    )

134810 équipements dans l'agglo (sur 1202 communes)
TYPEQU
A504    9622
D281    5909
A403    5465
A402    5283
A404    5197
        ... 
A105       1
A136       1
C505       1
D105       1
C304       1
Name: count, Length: 233, dtype: int64
26251 carreaux actifs conservés sur 26251 (population ou équipements)
4173 carreaux au-dessus du seuil (15.9% des carreaux) pour Services pour les particuliers (4.3, soit 100% de la moyenne)
2800 carreaux au-dessus du seuil (10.7% des carreaux) pour Commerces (2.6, soit 100% de la moyenne)
1871 carreaux au-dessus du seuil (7.1% des carreaux) pour Enseignement (0.8, soit 100% de la moyenne)
2263 carreaux au-dessus du seuil (8.6% des carreaux) pour Santé et action sociale (4.7, soit 100% de la moyenne)
1039 carreaux au-dessus du seuil (4.0% des carreaux) pour Transports et déplacements (0.1, soit 100% de la moyenne)
3199 carreaux au-dessus du seuil (12.2% des carreaux) pour Sports, loisirs et culture (1.2, soit 100% de la moyenne)
0 carreaux au-dessus

In [ ]:
#analyse BPE 1.2

# Un sous-dossier par réseau (nom_reseau_str) sous output/, pour ne pas mélanger
# les exports de plusieurs agglomérations dans le même dossier.
output_path_reseau = os.path.join(output_path, nom_reseau_str)
os.makedirs(output_path_reseau, exist_ok=True)

# Exemple d'usage : une carte pour un domaine
carte_ponderation_domaine(DOMAINES_BPE, population_grid_region, BPE_region, land_use_data, "D", tiles=FONDS_CARTE[FOND_CARTE])

#pour exporter toutes les cartes en HTML :
for d, nom_domaine in DOMAINES_BPE.items():
    carte = carte_ponderation_domaine(DOMAINES_BPE, population_grid_region, BPE_region, land_use_data, d, tiles=FONDS_CARTE[FOND_CARTE])

    # Titre au-dessus de la carte : .explore() n'a pas de paramètre title, on
    # l'ajoute donc en HTML directement dans le document folium (titre_carte_html,
    # cf. src/cartographie.py : position fixed, pour rester visible au-dessus du
    # div Leaflet plein cadre plutôt qu'être recouvert par lui). Nom explicite du
    # domaine (DOMAINES_BPE) plutôt que juste la lettre, pour un titre lisible.
    carte.get_root().html.add_child(
        folium.Element(titre_carte_html(f"Pondération {nom_domaine} – {nom_reseau_str}"))
    )

    nom_fichier_carte_d = f"ponderation_{d}_{nom_reseau_str}.html"
    carte.save(os.path.join(output_path_reseau, nom_fichier_carte_d))

    # Cache sur le dataset HF régional (cf. src.hf_cache) : mêmes chemins
    # relatifs "output/{nom_reseau_str}/..." que dans output_path_reseau en
    # local, pour que la structure distante corresponde à la structure locale.
    # Best-effort (silencieux si HF_TOKEN absent), ne bloque jamais le notebook.
    envoyer_vers_hf(
        os.path.join(output_path_reseau, nom_fichier_carte_d),
        f"output/{nom_reseau_str}/{nom_fichier_carte_d}",
        repo_id=HF_DATA_REPO_ID_REGION,
    )
  
# Affichage de la carte du domaine C (Enseignement) directement dans le notebook
from IPython.display import IFrame
IFrame(os.path.join(output_path_reseau, f"ponderation_C_{nom_reseau_str}.html"), width=900, height=600)

# Somme de la pondération (poids_gamme cumulé), totale et restreinte aux
# carreaux "pôles" (> seuil par domaine, cf. SEUILS_DOMAINE / land_use_data_domaine
# et pole_equipements_{domaine} en cellule "analyse BPE 1.1"), par domaine BPE.

from IPython.display import HTML, display

tableau_ponderation_domaine = pd.DataFrame(
    [
        {
            "Domaine": nom,
            "Pondération totale carreaux actifs": float(land_use_data_domaine(BPE_region, land_use_data, d)[d].sum()),
            "Pondération carreaux > seuils": float(
                land_use_data_domaine(BPE_region, land_use_data, d)
                .loc[land_use_data[f"pole_equipements_{d}"] == 1, d]
                .sum()
            ),
        }
        for d, nom in DOMAINES_BPE.items()
    ]
).set_index("Domaine")

tableau_ponderation_domaine_html = (
    tableau_ponderation_domaine.style.format("{:.1f}")
    .set_caption(f"Pondération des équipements par domaine – {nom_reseau_str}")
    .to_html()
)

display(HTML(tableau_ponderation_domaine_html))

chemin_tableau_ponderation = os.path.join(
    output_path_reseau, f"tableau_ponderation_domaine_{nom_reseau_str}.html"
)
with open(chemin_tableau_ponderation, "w", encoding="utf-8") as f:
    f.write(tableau_ponderation_domaine_html)

In [9]:
#Retourne différentes dates 
#dates_service, date_debut , date_fin , date_JOB = dates_service(feed)

dates_service_list, date_debut, date_fin, date_JOB = dates_service(feed)


In [ ]:
# Construction du réseau de transport multimodal, équivalent de setup_r5(data_path).
# En r5py, il n'y a pas de connexion séparée type r5r_core : l'objet TransportNetwork
# joue à la fois le rôle du réseau construit et du point d'entrée pour les calculs
# (ex: TravelTimeMatrixComputer, utilisé ensuite pour la matrice de temps de trajet).



print(data_path)
dir_tree(data_path)

# Vérifie D'ABORD si un ttm récent est déjà en cache (même test que dans la
# cellule "ttm" suivante, dupliqué ici) : si oui, on saute complètement la
# construction du réseau. Sans ce test, transport_network était TOUJOURS
# construit (extrait OSM régional + JVM r5py chargeant tout le graphe
# rues+transit de la région) même quand la cellule suivante allait de toute
# façon ignorer transport_network et recharger le ttm depuis le cache —
# l'objet restait résident en mémoire pour le reste de la session malgré
# tout, s'ajoutant au calcul des indicateurs plus loin. Cause des derniers
# kernels morts sur la VM malgré le passage de calculer_index_benchmark à sa
# version par lots (RAM cumulée réseau + calcul, pas juste le calcul seul).
os.makedirs(MEMORY_TTM_DIR, exist_ok=True)
TTM_PATH = os.path.join(MEMORY_TTM_DIR, f"ttm_{nom_reseau_str}.parquet")
recuperer_depuis_hf(f"memory_ttm/ttm_{nom_reseau_str}.parquet", TTM_PATH)
ttm_cache_recent = (
    os.path.exists(TTM_PATH) and (time.time() - os.path.getmtime(TTM_PATH)) < 10 * 24 * 3600
)

if ttm_cache_recent:
    print(f"ttm déjà en cache (< 10 jours) — réseau non reconstruit : {TTM_PATH}")
    transport_network = None
else:
    # Extrait OSM régional (Geofabrik, cf. src/build_data_agglo.py) : à
    # l'échelle d'une région entière, un seul fichier pré-découpé par
    # Geofabrik plutôt que le découpage en tuiles Overpass d'osm_pbf_creator
    # (pensé pour une emprise de taille agglo, cf.
    # telecharger_osm_pbf_geofabrik pour le détail).
    from src.build_data_agglo import telecharger_osm_pbf_geofabrik
    from src.utils import preparer_gtfs_pour_r5py

    OSM_PBF_PATH = os.path.join(data_path, f"region_{NOM_REGION_STR}.osm.pbf")
    telecharger_osm_pbf_geofabrik(NOM_REGION_STR, OSM_PBF_PATH)

    # preparer_gtfs_pour_r5py : retire du GTFS les tables optionnelles
    # présentes mais vides, que le lecteur GTFS de r5py rejette sinon avec
    # une EmptyTableError (cf. sa docstring, src/utils.py).
    gtfs_r5py_path = preparer_gtfs_pour_r5py(GTFS_PATH)

    # allow_errors=True : sans ce paramètre, le chargement plante avec
    # "java.lang.ArrayIndexOutOfBoundsException: Attempt to seek beyond end
    # of edge store" (StreetLayer.loadFromOsm) sur cet extrait régional
    # pourtant valide (vérifié avec `osmium fileinfo` : taille et structure
    # PBF correctes) — R5 refuse une way qu'il juge problématique quelque
    # part dans l'extrait plutôt que de l'ignorer. allow_errors=True tolère
    # ce type d'erreur au lieu de faire échouer tout le chargement du réseau.
    transport_network = r5py.TransportNetwork(
        OSM_PBF_PATH, gtfs=[str(gtfs_r5py_path)], allow_errors=True
    )

In [ ]:
# Points d'origine/destination = centroïdes de la grille de population,
# équivalent de points <- fread(file.path(data_path, "poa_hexgrid.csv")).
# r5py exige une géométrie de type Point (pas les carreaux polygones bruts)
# et au moins une colonne "id", déjà présente dans population_grid_region.
points = population_grid_region[["id", "geometry"]].copy()
points["geometry"] = points.geometry.centroid

os.makedirs(MEMORY_TTM_DIR, exist_ok=True)
TTM_PATH = os.path.join(MEMORY_TTM_DIR, f"ttm_{nom_reseau_str}.parquet")

# Repli sur le cache Hugging Face si absent en local (cf. src/hf_cache.py) :
# no-op silencieux si déjà présent, ou si absent des deux côtés. Le fichier
# téléchargé a une date de modification récente, donc compte comme "cache
# récent" pour le test ttm_cache_recent ci-dessous.
recuperer_depuis_hf(f"memory_ttm/ttm_{nom_reseau_str}.parquet", TTM_PATH)

# Si un ttm a déjà été calculé pour ce réseau il y a moins de 10 jours, on le
# recharge directement plutôt que de relancer TravelTimeMatrix (calcul long).
# Au-delà de 10 jours, on considère le GTFS potentiellement obsolète (nouveau
# service, changement d'horaires) et on préfère recalculer.
ttm_cache_recent = (
    os.path.exists(TTM_PATH) and (time.time() - os.path.getmtime(TTM_PATH)) < 10 * 24 * 3600
)

if ttm_cache_recent:
    print(f"ttm déjà en cache (< 10 jours), pas de recalcul : {TTM_PATH}")
else:
    # date_JOB (calculé en cellule "#Retourne différentes dates" à partir des
    # dates de service réellement présentes dans le GTFS) plutôt qu'une date
    # fixe arbitraire, qui n'aurait aucune raison de tomber dans la période
    # de validité du GTFS régional.
    departure_datetime = datetime.datetime.strptime(date_JOB, "%Y%m%d").replace(
        hour=14, minute=0, second=0
    )

    # Par lots d'origines (calculer_ttm_par_lots, src.utilitaires_matrix) plutôt
    # qu'un seul appel origins=destinations=tous les carreaux : une région
    # entière (Bretagne : 26 251 carreaux 1km) est assez grosse pour qu'un
    # calcul en un seul bloc dépasse largement la RAM disponible. Le résultat
    # est écrit sur disque au fur et à mesure, lot par lot, sans jamais garder
    # la matrice complète en mémoire Python pendant le calcul.
    calculer_ttm_par_lots(
        r5py,
        transport_network,
        points,
        departure=departure_datetime,
        transport_modes=[r5py.TransportMode.WALK, r5py.TransportMode.TRANSIT],
        max_time_walking=datetime.timedelta(minutes=30),
        # 90 min : cutoff le plus large retenu pour l'analyse régionale
        # (CUTOFFS_MINUTES = (60, 90) ci-dessous) — inutile de calculer des
        # temps de trajet au-delà de ce qui sera analysé, ça alourdirait la
        # matrice (et sa taille en mémoire/disque) pour rien.
        max_time=datetime.timedelta(minutes=90),
        ttm_path=TTM_PATH,
        on_step=print,
    )

    # Renvoi vers le cache Hugging Face partagé avec l'app (cf. src.hf_cache) :
    # sans ça, ce calcul (fait ici, en local, pour disposer de plus de RAM que le
    # Space sur les grosses régions) resterait invisible pour l'app, qui le
    # recalculerait elle-même — best-effort, ne bloque jamais le notebook.
    envoyer_vers_hf(TTM_PATH, f"memory_ttm/ttm_{nom_reseau_str}.parquet")

# Pas de ttm = charger_ttm(TTM_PATH) ici : rien en aval ne charge plus le
# ttm complet en mémoire (cellules d'indicateurs/cartes suivantes, lisent
# TTM_PATH par lots via calculer_index_benchmark_par_lots/
# cost_to_closest_par_lots/cumulative_cutoff_par_lots) — inutile de payer le
# coût mémoire d'un chargement complet (quelques Go) pour rien.
print(f"TTM_PATH prêt : {TTM_PATH}")

In [ ]:
# Analyse accessibilité régionale : % des équipements pondérés (par domaine
# BPE) accessible en <= 60 min et <= 90 min en transport en commun + marche,
# moyenne pondérée par la population du carreau d'origine.
#
# calculer_index_benchmark_par_lots (pas calculer_index_benchmark) : lit
# TTM_PATH par row group pyarrow plutôt que de charger le ttm entier en
# mémoire — calculer_index_benchmark a déjà fait exploser la RAM à l'échelle
# région (~20 Go en quelques secondes, process tué, mesuré sur la Bretagne
# le 29/08) : même plafond mémoire déjà documenté sur IDFM (32 Go dépassés)
# dans sa docstring. Résultat numériquement identique (vérifié sur un jeu de
# données synthétique), cf. la docstring de calculer_index_benchmark_par_lots
# (src/utilitaires_matrix.py).
#
# cutoffs=(60, 90) : les deux seuils d'accès demandés pour cette analyse
# régionale (au lieu du défaut (30, 45, 60), pensé pour une échelle agglo où
# les trajets sont plus courts) — cohérent avec max_time=90 min retenu pour
# le calcul du ttm ci-dessus (aucun trajet > 90 min dans la matrice de toute
# façon, donc un cutoff au-delà de 90 min serait tronqué silencieusement).
CUTOFFS_MINUTES = (60, 90)

niveau_vie = deciles_niveau_vie(population_grid_region)

index_accessibilite_region = calculer_index_benchmark_par_lots(
    TTM_PATH,
    BPE_region,
    land_use_data,
    DOMAINES_BPE,
    niveau_vie,
    cutoffs=CUTOFFS_MINUTES,
    on_step=print,
)

CHEMIN_INDEX_ACCESSIBILITE = os.path.join(
    output_path_reseau, f"index_accessibilite_{nom_reseau_str}.csv"
)
exporter_df_to_csv(index_accessibilite_region, CHEMIN_INDEX_ACCESSIBILITE)
envoyer_vers_hf(
    CHEMIN_INDEX_ACCESSIBILITE,
    f"output/{nom_reseau_str}/index_accessibilite_{nom_reseau_str}.csv",
    repo_id=HF_DATA_REPO_ID_REGION,
)

# Vue synthétique : tous carreaux confondus ("Tous", pas de décile), un
# domaine par ligne, % moyen pondéré par la population accessible à 60 et 90 min.
index_accessibilite_region.loc[
    index_accessibilite_region["decile"] == "Tous",
    ["nom_domaine", "pct_equipement_pondere_60min", "pct_equipement_pondere_90min"],
].set_index("nom_domaine")

In [ ]:
# --- Carte du temps d'accès au pôle d'équipements le plus proche, par domaine ---
#
# Restaurée après suppression (cellules 9.1/9.2 du notebook agglo original,
# retirées le 29/08 faute d'être mémoire-safe à l'échelle région — elles
# opéraient sur le ttm entier chargé en mémoire) : réécrite ici en version
# par lots (cost_to_closest_par_lots, cf. src/utilitaires_matrix.py), qui
# lit TTM_PATH row group par row group plutôt que de charger le ttm complet.
# Seuils 60/90 min (pas 30/45 comme dans le notebook agglo original) pour
# rester cohérent avec le reste de ce notebook région (CUTOFFS_MINUTES).
#
# ⚠ Coût : une passe complète sur TTM_PATH par domaine (8 domaines), donc
# nettement plus long que le calcul des indicateurs agrégés
# (calculer_index_benchmark_par_lots, une seule passe pour tous les domaines
# à la fois) — attendre plusieurs (dizaines de) minutes selon la taille du
# ttm régional.

min_time_par_domaine = {}

for d, nom_domaine in DOMAINES_BPE.items():
    # Restreint aux carreaux "pôles" (pole_equipements_{d}) plutôt qu'à
    # n'importe quel carreau ayant au moins 1 unité d'équipement.
    poles_d = land_use_data[["id", f"pole_equipements_{d}"]].rename(columns={f"pole_equipements_{d}": d})

    min_time_d = cost_to_closest_par_lots(TTM_PATH, poles_d, d, "travel_time", on_step=print)
    min_time_par_domaine[d] = min_time_d

    # Plafonné à 90 min (le max_time retenu pour tout le ttm régional, cf.
    # cellule "ttm" plus haut) pour la lisibilité de la carte : au-delà,
    # seule l'idée de "trop loin" compte (valeur "inf", aucun trajet trouvé
    # sous 90 min), pas la valeur exacte.
    carte_temps = population_grid_region[["id", "geometry"]].merge(min_time_d, on="id")
    carte_temps["travel_time_plafonne"] = carte_temps["travel_time"].clip(upper=90)

    fig, ax = plt.subplots(figsize=(8, 8))
    carte_temps.to_crs(epsg=3857).plot(
        column="travel_time_plafonne",
        cmap="cividis_r",
        alpha=0.9,
        legend=False,
        edgecolor="none",
        ax=ax,
    )

    # Colorbar calée sur la hauteur réelle de la carte : legend=True
    # donnerait une colorbar de la hauteur de la figure entière, plus grande
    # que la carte une fois l'aspect ratio géographique appliqué.
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    fig.colorbar(ax.collections[0], cax=cax, label="Temps (min)")

    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs="EPSG:3857")
    ax.set_axis_off()
    fig.suptitle(f"Temps d'accès au pôle d'équipements le plus proche – {nom_domaine} – {nom_reseau_str}")

    chemin_base_9_1 = os.path.join(output_path_reseau, f"accessibilite_temps_{d}_{nom_reseau_str}")
    plt.savefig(f"{chemin_base_9_1}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Export HTML interactif équivalent (contours des carreaux transparents,
    # cf. carte_ponderation_domaine).
    carte_interactive_9_1 = carte_temps.explore(
        column="travel_time_plafonne",
        cmap="cividis_r",
        tiles=FONDS_CARTE[FOND_CARTE],
        legend=False,  # légende maison ci-dessous (echelle_continue_html) :
        # .explore(legend=True) positionne en haut à droite (branca), ce qui
        # chevauche le titre flottant (titre_carte_html, en haut).
        style_kwds={"weight": 0, "opacity": 0},
    )
    carte_interactive_9_1.get_root().html.add_child(
        folium.Element(titre_carte_html(
            f"Temps d'accès au pôle d'équipements le plus proche – {nom_domaine} – {nom_reseau_str}"
        ))
    )
    carte_interactive_9_1.get_root().html.add_child(
        folium.Element(echelle_continue_html(
            carte_temps["travel_time_plafonne"].min(),
            carte_temps["travel_time_plafonne"].max(),
            "cividis_r",
            "Temps (min)",
            cote="centre",
        ))
    )

    minx_9_1, miny_9_1, maxx_9_1, maxy_9_1 = carte_temps.to_crs(epsg=4326).total_bounds
    carte_interactive_9_1.get_root().html.add_child(
        folium.Element(script_reajuster_si_masque(
            carte_interactive_9_1, [[miny_9_1, minx_9_1], [maxy_9_1, maxx_9_1]]
        ))
    )

    nom_fichier_9_1 = f"accessibilite_temps_{d}_{nom_reseau_str}.html"
    carte_interactive_9_1.save(os.path.join(output_path_reseau, nom_fichier_9_1))

    # Cache sur le dataset HF régional (best-effort, cf. src.hf_cache).
    envoyer_vers_hf(
        os.path.join(output_path_reseau, nom_fichier_9_1),
        f"output/{nom_reseau_str}/{nom_fichier_9_1}",
        repo_id=HF_DATA_REPO_ID_REGION,
    )

In [ ]:
# --- Carte du nombre de pôles d'équipements accessibles, par domaine (60 vs 90 min) ---
#
# Restaurée après suppression, version par lots (cumulative_cutoff_par_lots)
# — cf. la cellule précédente pour le détail. ⚠ Coût similaire : une passe
# complète sur TTM_PATH par (domaine x cutoff), 8 domaines x 2 cutoffs.

from folium.plugins import DualMap

cum_60_par_domaine = {}
cum_90_par_domaine = {}

for d, nom_domaine in DOMAINES_BPE.items():
    # Restreint aux carreaux "pôles" (pole_equipements_{d}) : cum_60_d/cum_90_d
    # comptent le nombre de pôles accessibles, pas une somme pondérée continue
    # sur tous les carreaux ayant de l'équipement.
    poles_d = land_use_data[["id", f"pole_equipements_{d}"]].rename(columns={f"pole_equipements_{d}": d})

    cum_60_d = cumulative_cutoff_par_lots(TTM_PATH, poles_d, d, "travel_time", cutoff=60, on_step=print)
    cum_90_d = cumulative_cutoff_par_lots(TTM_PATH, poles_d, d, "travel_time", cutoff=90, on_step=print)
    cum_60_par_domaine[d] = cum_60_d
    cum_90_par_domaine[d] = cum_90_d

    limite_commune = max(cum_60_d[d].max(), cum_90_d[d].max())

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    for ax, cum, titre in zip(axes, [cum_60_d, cum_90_d], ["Jusqu'à 60 min", "Jusqu'à 90 min"]):
        carte = population_grid_region[["id", "geometry"]].merge(cum, on="id").to_crs(epsg=3857)
        carte.plot(
            column=d, cmap="inferno", vmin=0, vmax=limite_commune, alpha=0.9,
            legend=False, edgecolor="none", ax=ax,
        )

        # Colorbar calée sur la hauteur réelle de chaque carte (cf. cellule
        # précédente) : legend=True donnerait une colorbar de la hauteur de
        # la figure entière, plus grande que la carte une fois l'aspect
        # ratio géographique appliqué.
        from mpl_toolkits.axes_grid1 import make_axes_locatable

        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(ax.collections[0], cax=cax, label="Pôles d'équipements accessibles")

        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, crs="EPSG:3857")
        ax.set_axis_off()
        ax.set_title(titre)
    fig.suptitle(f"Pôles d'équipements accessibles – {nom_domaine} – {nom_reseau_str}")
    plt.tight_layout()

    chemin_base_9_2 = os.path.join(output_path_reseau, f"accessibilite_cumule_{d}_{nom_reseau_str}")
    plt.savefig(f"{chemin_base_9_2}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Export HTML interactif : DualMap (folium.plugins) = deux cartes synchronisées
    # côte à côte (60 min à gauche, 90 min à droite) dans un seul fichier HTML,
    # même principe que le PNG ci-dessus mais interactif.
    carte_60 = population_grid_region[["id", "geometry"]].merge(cum_60_d, on="id")
    carte_90 = population_grid_region[["id", "geometry"]].merge(cum_90_d, on="id")

    dual_map = DualMap(tiles=FONDS_CARTE[FOND_CARTE], layout="horizontal")
    # legend=False sur les deux : branca cible tous les colorbars de la page
    # via un sélecteur CSS non isolé par carte — le second colorbar
    # s'empile dans le premier au lieu de s'afficher sur son propre panneau.
    # Légendes maison à la place (echelle_continue_html), une par côté.
    carte_60.explore(
        column=d, cmap="inferno", vmin=0, vmax=limite_commune,
        legend=False,
        style_kwds={"weight": 0, "opacity": 0}, m=dual_map.m1,
    )
    carte_90.explore(
        column=d, cmap="inferno", vmin=0, vmax=limite_commune,
        legend=False,
        style_kwds={"weight": 0, "opacity": 0}, m=dual_map.m2,
    )

    # .explore(m=...) ajoute la couche à une carte existante sans recentrer
    # dessus : DualMap() démarre donc sur sa vue par défaut ([0, 0], zoom 1)
    # sans ce fit_bounds explicite sur les deux volets.
    minx, miny, maxx, maxy = carte_60.to_crs(epsg=4326).total_bounds
    dual_map.m1.fit_bounds([[miny, minx], [maxy, maxx]])
    dual_map.m2.fit_bounds([[miny, minx], [maxy, maxx]])

    dual_map.get_root().html.add_child(
        folium.Element(titre_carte_html(
            f"Pôles d'équipements accessibles – {nom_domaine} – {nom_reseau_str} (60 min / 90 min)"
        ))
    )
    dual_map.get_root().html.add_child(
        folium.Element(echelle_continue_html(0, limite_commune, "inferno", f"{nom_domaine} (pôles) – 60 min", cote="gauche"))
    )
    dual_map.get_root().html.add_child(
        folium.Element(echelle_continue_html(0, limite_commune, "inferno", f"{nom_domaine} (pôles) – 90 min", cote="droite"))
    )
    dual_map.get_root().html.add_child(
        folium.Element(script_reajuster_si_masque(dual_map.m1, [[miny, minx], [maxy, maxx]]))
    )
    dual_map.get_root().html.add_child(
        folium.Element(script_reajuster_si_masque(dual_map.m2, [[miny, minx], [maxy, maxx]]))
    )

    nom_fichier_9_2 = f"accessibilite_cumule_{d}_{nom_reseau_str}.html"
    dual_map.save(os.path.join(output_path_reseau, nom_fichier_9_2))

    # Cache sur le dataset HF régional (best-effort, cf. src.hf_cache).
    envoyer_vers_hf(
        os.path.join(output_path_reseau, nom_fichier_9_2),
        f"output/{nom_reseau_str}/{nom_fichier_9_2}",
        repo_id=HF_DATA_REPO_ID_REGION,
    )